# Human Gene Essentiality V2 - FINAL FIX
## Expression file has ModelID column for cell line matching

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded!")

In [ ]:
DATA_DIR = '/content/drive/MyDrive/depmap_data'
print("Files:")
for f in os.listdir(DATA_DIR):
    print(f"  {f}")

In [ ]:
# Load CRISPR - this one is straightforward
print("Loading CRISPR...")
crispr_raw = pd.read_csv(os.path.join(DATA_DIR, 'CRISPRGeneEffect.csv'), index_col=0)
print(f"Shape: {crispr_raw.shape}")
print(f"Index (cell lines): {crispr_raw.index[:3].tolist()}")
print(f"Columns (genes): {crispr_raw.columns[:3].tolist()}")

In [ ]:
# Load Expression - need to handle differently
print("\nLoading Expression...")

# Find expression file
expr_file = None
for f in os.listdir(DATA_DIR):
    if 'expression' in f.lower() and 'crispr' not in f.lower():
        expr_file = f
        break
print(f"Expression file: {expr_file}")

# Read first few rows to understand structure
expr_preview = pd.read_csv(os.path.join(DATA_DIR, expr_file), nrows=5)
print(f"\nFirst 10 columns: {expr_preview.columns[:10].tolist()}")
print(f"\nFirst few rows of first 5 columns:")
print(expr_preview.iloc[:, :5])

In [ ]:
# The expression file has metadata columns first, then gene columns
# ModelID column contains the cell line ID (ACH-XXXXXX)

print("Loading full expression data...")
expr_full = pd.read_csv(os.path.join(DATA_DIR, expr_file))
print(f"Shape: {expr_full.shape}")

# Find the ModelID column (contains ACH-XXXXXX)
model_id_col = None
for col in expr_full.columns[:10]:
    if expr_full[col].dtype == 'object':
        sample = str(expr_full[col].iloc[0])
        if sample.startswith('ACH-'):
            model_id_col = col
            print(f"Found cell line ID column: '{col}'")
            print(f"Sample values: {expr_full[col].iloc[:3].tolist()}")
            break

if model_id_col is None:
    # Try looking at all columns
    print("\nSearching all columns for ACH- pattern...")
    for col in expr_full.columns:
        if expr_full[col].dtype == 'object':
            try:
                if str(expr_full[col].iloc[0]).startswith('ACH-'):
                    model_id_col = col
                    print(f"Found: '{col}'")
                    break
            except:
                pass

In [ ]:
# Set ModelID as index and drop metadata columns
if model_id_col:
    # Find where gene columns start (first column with parentheses like 'GENE (12345)')
    gene_start_idx = 0
    for i, col in enumerate(expr_full.columns):
        if '(' in str(col) and ')' in str(col):
            gene_start_idx = i
            print(f"Gene columns start at index {i}: '{col}'")
            break
    
    # Create clean expression dataframe
    expr_clean = expr_full.iloc[:, gene_start_idx:].copy()
    expr_clean.index = expr_full[model_id_col].values
    
    print(f"\nCleaned expression shape: {expr_clean.shape}")
    print(f"Index (cell lines): {list(expr_clean.index[:3])}")
    print(f"Columns (genes): {list(expr_clean.columns[:3])}")
else:
    print("ERROR: Could not find ModelID column!")

In [ ]:
# Now match cell lines
crispr_cells = set(crispr_raw.index)
expr_cells = set(expr_clean.index)
common_cells = list(crispr_cells & expr_cells)

print(f"CRISPR cell lines: {len(crispr_cells)}")
print(f"Expression cell lines: {len(expr_cells)}")
print(f"Common cell lines: {len(common_cells)}")

In [ ]:
# Match genes (extract symbol from 'GENE (12345)' format)
def get_symbol(name):
    if ' (' in str(name):
        return str(name).split(' (')[0].upper()
    return str(name).upper()

crispr_sym2col = {get_symbol(c): c for c in crispr_raw.columns}
expr_sym2col = {get_symbol(c): c for c in expr_clean.columns}

common_symbols = set(crispr_sym2col.keys()) & set(expr_sym2col.keys())
print(f"Common gene symbols: {len(common_symbols)}")

In [ ]:
# Create aligned dataframes
print("Creating aligned dataframes...")

crispr_cols = [crispr_sym2col[s] for s in common_symbols]
expr_cols = [expr_sym2col[s] for s in common_symbols]

# Subset and align
crispr_aligned = crispr_raw.loc[common_cells, crispr_cols].copy()
expr_aligned = expr_clean.loc[common_cells, expr_cols].copy()

# Use common symbols as column names
crispr_aligned.columns = list(common_symbols)
expr_aligned.columns = list(common_symbols)

# Transpose: genes x cells
crispr_df = crispr_aligned.T
expr_df = expr_aligned.T

print(f"CRISPR aligned: {crispr_df.shape}")
print(f"Expression aligned: {expr_df.shape}")

# Verify
assert list(crispr_df.index) == list(expr_df.index)
assert list(crispr_df.columns) == list(expr_df.columns)
print("✅ Alignment verified!")

In [ ]:
# Binarize
THRESHOLD = -0.5
binary_mat = (crispr_df < THRESHOLD).astype(int)
print(f"Binary matrix: {binary_mat.shape}")
print(f"Essential: {binary_mat.sum().sum():,} ({100*binary_mat.sum().sum()/binary_mat.size:.1f}%)")

In [ ]:
# Gene classification
gene_consensus = binary_mat.mean(axis=1)

pan_mask = gene_consensus >= 0.9
non_mask = gene_consensus <= 0.1
context_mask = ~pan_mask & ~non_mask

print(f"Pan-essential: {pan_mask.sum()}")
print(f"Non-essential: {non_mask.sum()}")
print(f"Context-dependent: {context_mask.sum()}")

In [ ]:
# V2 Predictor
class HedgeFundV2:
    def __init__(self):
        self.ml = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1)
        self.scaler = StandardScaler()
        self.fitted = False
        
    def _feat(self, bmat, emat, gene, cell, others):
        cons = bmat.loc[gene, others].mean()
        var = bmat.loc[gene, others].var()
        expr = emat.loc[gene, cell]
        expr_m = emat.loc[gene, others].mean()
        expr_s = emat.loc[gene, others].std() + 1e-6
        expr_z = (expr - expr_m) / expr_s
        expr_p = (emat.loc[gene, :] < expr).mean()
        return [cons, var, expr, expr_z, expr_p, cons*expr, cons**2, expr**2]
    
    def fit(self, bmat, emat, n_train=100):
        print("Training V2...")
        cons = bmat.mean(axis=1)
        ctx = cons[(cons > 0.1) & (cons < 0.9)].index.tolist()
        print(f"  Context genes: {len(ctx)}")
        
        cells = list(bmat.columns)[:n_train]
        X, y = [], []
        
        for c in tqdm(cells, desc="  Training"):
            others = [x for x in bmat.columns if x != c]
            for g in ctx:
                try:
                    f = self._feat(bmat, emat, g, c, others)
                    if not np.isnan(f).any():
                        X.append(f)
                        y.append(bmat.loc[g, c])
                except: pass
        
        X, y = np.array(X), np.array(y)
        print(f"  Samples: {len(X):,}, Pos rate: {y.mean():.1%}")
        
        Xs = self.scaler.fit_transform(X)
        self.ml.fit(Xs, y)
        
        yp = self.ml.predict(Xs)
        print(f"  Train acc: {accuracy_score(y, yp):.3f}, bal: {balanced_accuracy_score(y, yp):.3f}")
        
        names = ['cons', 'var', 'expr', 'expr_z', 'expr_p', 'c*e', 'c^2', 'e^2']
        print("  Importance:", {n: f"{i:.2f}" for n, i in zip(names, self.ml.feature_importances_)})
        
        self.fitted = True
        return self
    
    def predict(self, bmat, emat, cell):
        others = [c for c in bmat.columns if c != cell]
        cons = bmat[others].mean(axis=1)
        preds = pd.Series(0, index=bmat.index)
        preds[cons >= 0.9] = 1
        
        ctx = (cons > 0.1) & (cons < 0.9)
        ctx_genes = preds[ctx].index.tolist()
        
        if ctx_genes and self.fitted:
            X, valid = [], []
            for g in ctx_genes:
                try:
                    f = self._feat(bmat, emat, g, cell, others)
                    if not np.isnan(f).any():
                        X.append(f)
                        valid.append(g)
                except: pass
            if X:
                Xs = self.scaler.transform(np.array(X))
                for g, p in zip(valid, self.ml.predict(Xs)):
                    preds[g] = p
        return preds

print("V2 defined!")

In [ ]:
# Train
pred = HedgeFundV2()
pred.fit(binary_mat, expr_df, n_train=100)

In [ ]:
# Evaluate
N = 50
cells = list(binary_mat.columns)
test = cells[100:100+N]

yt, yp = [], []
for c in tqdm(test, desc="Testing"):
    yt.extend(binary_mat[c].values)
    yp.extend(pred.predict(binary_mat, expr_df, c).values)

yt, yp = np.array(yt), np.array(yp)
print(f"\nAccuracy: {accuracy_score(yt, yp):.3f}")
print(f"Balanced: {balanced_accuracy_score(yt, yp):.3f}")
print(f"F1: {f1_score(yt, yp):.3f}")

In [ ]:
# By category
v1 = {'Pan-essential (≥90%)': 0.982, 'Common (50-90%)': 0.434, 
      'Context-dep (10-50%)': 0.439, 'Rarely ess (0-10%)': 0.991, 'Non-essential (0%)': 1.0}

cats = {
    'Pan-essential (≥90%)': gene_consensus >= 0.9,
    'Common (50-90%)': (gene_consensus >= 0.5) & (gene_consensus < 0.9),
    'Context-dep (10-50%)': (gene_consensus >= 0.1) & (gene_consensus < 0.5),
    'Rarely ess (0-10%)': (gene_consensus > 0) & (gene_consensus < 0.1),
    'Non-essential (0%)': gene_consensus == 0
}

print("\nV1 → V2:")
print("-"*70)
for name, mask in cats.items():
    n = mask.sum()
    if n > 0:
        m = np.tile(mask.values, N)
        v2_acc = accuracy_score(yt[m], yp[m])
        v1_acc = v1[name]
        d = v2_acc - v1_acc
        e = "✅" if d > 0.05 else "➡️" if d > -0.05 else "❌"
        print(f"{name:25s}: {v1_acc:.1%} → {v2_acc:.1%} ({d:+.1%}) {e}")

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(cats))
w = 0.35

v1_vals = [v1[c] for c in cats]
v2_vals = [accuracy_score(yt[np.tile(m.values, N)], yp[np.tile(m.values, N)]) for m in cats.values()]

ax.bar(x - w/2, v1_vals, w, label='V1', color='#ff6b6b')
ax.bar(x + w/2, v2_vals, w, label='V2', color='#4ecdc4')
ax.axhline(0.5, color='gray', ls='--')
ax.set_xticks(x)
ax.set_xticklabels(cats.keys(), rotation=45, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('V1 vs V2')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()